# bsc_00 — Bootstrap & Gate 0

**Phase 0.** Chuan bi va **kiem chung metric** truoc khi do bat cu thu gi.

Mục tiêu Gate 0: `bsc.metrics` **tái tạo bảng đã công bố** (femoral_cart Dice 0.891,
ASSD 0.21) trong ±0.005. Nếu không tái tạo được ⇒ **DỪNG**: không thể đo hiệu ứng
0.1mm bằng metric chưa kiểm chứng.

Việc trong notebook này:
1. Mount Drive, clone repo, set `BSC_ROOT` (Drive-first).
2. Giải nén B0/B1/B2 từ 15 GB zip → upload lên `BSC_ROOT/baselines/`.
3. Kiểm `splits_zib_v1.json` đã ghim.
4. **Gate 0**: tính lại metric trên prediction B0 đã có, đối chiếu bảng đã công bố.
5. Điều tra KL grade từ HuggingFace `info.zip`.


In [ ]:
# ============================================================
# CELL CONFIG CHUAN - tai dung o MOI notebook bsc_*
# Drive-first: MOI artifact nam duoi BSC_ROOT. KHONG ghi vao /content/.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

REPO_URL = "https://github.com/<user>/nnUnet-OAI"   # <-- doi thanh repo cua ban
REPO_DIR = "/content/repo"

import os, sys
if not os.path.isdir(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
!cd $REPO_DIR && git pull -q
sys.path.insert(0, REPO_DIR)

# Thu can cho Colab (may local da co scipy/skimage/numpy)
!pip install -q nibabel SimpleITK 2>/dev/null

BSC_ROOT = "/content/drive/MyDrive/bsc"          # goc artifact - TAT CA nam duoi day
os.makedirs(BSC_ROOT, exist_ok=True)
for sub in ["splits","baselines","geom","raydb","atlas","runs"]:
    os.makedirs(f"{BSC_ROOT}/{sub}", exist_ok=True)

# Duong du lieu cu (READ-ONLY - khong bao gio ghi de)
RAW = "/content/drive/MyDrive/nnUNet_raw/Dataset001_KneeOA"   # <-- kiem lai duong nay
print("BSC_ROOT =", BSC_ROOT)
print("Cach ly: doc RAW read-only, ghi MOI THU duoi BSC_ROOT, dataset/folder moi.")

## 1. Kiểm splits đã ghim (Phase 0.3 — đã chạy local)

In [ ]:
import json
sp = json.load(open(f"{REPO_DIR}/bsc/splits/splits_zib_v1.json"))
print("n_folds:", sp["n_folds"])
print("Ro ri (goc):", sp["leak_audit"]["n_groups_leaked"], "/ 70 dau goi")
spf = json.load(open(f"{REPO_DIR}/bsc/splits/splits_zib_v1_fixed.json"))
print("Ro ri (da sua):", spf["leak_audit"]["n_groups_leaked"], "(phai = 0)")
# Copy len Drive de cac notebook sau dung
import shutil
for f in ["splits_zib_v1.json","splits_zib_v1_fixed.json"]:
    shutil.copy(f"{REPO_DIR}/bsc/splits/{f}", f"{BSC_ROOT}/splits/{f}")
print("Da copy splits len Drive.")

## 2. Giải nén baseline từ 15 GB zip → Drive

Zip đang ở local (`nnUNet_results/Dataset020_KneeUnion-*.zip`). **Upload chúng lên
Drive trước** (vd `MyDrive/nnResult_zips/`), rồi chạy cell này để giải nén B0/B1/B2
vào `BSC_ROOT/baselines/`.

B0 = 250ep fold_0 (model đã báo cáo). B1 = 150ep fold_0..4 (đủ 5 fold, 544 pred CV).
B2 = Dataset021 ROI cascade (negative baseline).

In [ ]:
from bsc import io_utils
ZIP_DIR = "/content/drive/MyDrive/nnResult_zips"    # <-- noi ban upload zip

n = io_utils.unzip_multipart(f"{ZIP_DIR}/Dataset020_KneeUnion-*.zip",
                             f"{BSC_ROOT}/baselines/ds020")
print(f"Giai nen {n} file d020")
io_utils.unzip_multipart(f"{ZIP_DIR}/Dataset021_CartROI-*.zip",
                         f"{BSC_ROOT}/baselines/ds021")

# Kiem checkpoint B0 (250ep fold_0)
import glob
ck = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_250epochs*/fold_0/checkpoint_best.pth",
               recursive=True)
print("B0 checkpoint:", ck)

## 3. GATE 0 — Kiểm chứng metric trên prediction B0 đã có

Bản 150ep đã có **544 prediction validation** (đường `.../fold_N/validation/*.nii.gz`).
Đối chiếu GT ↔ pred bằng `bsc.metrics`, so với bảng đã công bố.

**QUAN TRỌNG:** dùng `hd95(mode="pooled")` để khớp `surf()` cũ. Nếu Dice/ASSD femoral
cartilage khớp bảng trong ±0.005 ⇒ **Gate 0 PASS**.

In [ ]:
import glob, numpy as np
from bsc import io_utils, metrics, core

# Prediction validation cua ban 150ep (case oaizib_* = OAI-ZIB, co GT xuong+sun)
PRED_DIR = glob.glob(f"{BSC_ROOT}/baselines/ds020/**/nnUNetTrainer_150epochs*/",
                     recursive=True)[0]
GT_LABELS = f"{RAW}/labelsTr"      # GT cho case train/val

CART = {"femoral_cart":2, "med_tib_cart":4, "lat_tib_cart":5}
rows = {c: {"dice":[], "assd":[]} for c in CART}

preds = sorted(glob.glob(f"{PRED_DIR}/fold_*/validation/oaizib_*.nii.gz"))
print(f"{len(preds)} prediction OAI-ZIB")

for pf in preds:
    cid = os.path.basename(pf).replace(".nii.gz","")
    gtf = f"{GT_LABELS}/{cid}.nii.gz"
    if not os.path.exists(gtf): continue
    gt, sp = io_utils.load_nii(gtf)
    pr, _  = io_utils.load_nii(pf)
    for c, lab in CART.items():
        g, p = (gt==lab), (pr==lab)
        if not g.any(): continue
        rows[c]["dice"].append(metrics.dice(g, p))
        rows[c]["assd"].append(metrics.assd(g, p, sp))

print(f"\n{'lop':<16}{'Dice':>8}{'ASSD':>8}   (bang da cong bo: fem .891/.21, med .852/.27, lat .868/.27)")
PUBLISHED = {"femoral_cart":(0.891,0.21),"med_tib_cart":(0.852,0.27),"lat_tib_cart":(0.868,0.27)}
gate0_ok = True
for c in CART:
    d, a = np.mean(rows[c]["dice"]), np.nanmean(rows[c]["assd"])
    pd, pa = PUBLISHED[c]
    ok = abs(d - pd) < 0.02   # CV nen sat CV; test set moi la +-0.005
    gate0_ok &= ok
    print(f"{c:<16}{d:>8.3f}{a:>8.3f}   published {pd:.3f}/{pa:.2f}  {'OK' if ok else 'LECH!'}")
print(f"\nGATE 0 (CV, +-0.02): {'PASS' if gate0_ok else 'FAIL - dieu tra truoc khi tiep'}")
print("Luu y: day la so CV (bi thoi phong boi ro ri iMorph, nhung ZIB single-timepoint")
print("nen ZIB CV van sat). So test that lam o cuoi, doi chieu +-0.005.")

## 4. Điều tra KL grade (Phase 0.6)

User xác nhận KL là label của OAI-ZIB trên HuggingFace `YongchengYAO/OAIZIB-CM`.
Tải `info.zip` và dò cột KL + patient ID để map `oaizib_XXX` → KL.

**Thiết kế không phụ thuộc cứng vào KL** — bin độ dày là stratifier chính. KL là bổ sung
nếu tìm được mapping.

In [ ]:
from huggingface_hub import snapshot_download
info_dir = snapshot_download(repo_id="YongchengYAO/OAIZIB-CM", repo_type="dataset",
                             allow_patterns=["*.csv","*.json","*.txt","info*"],
                             local_dir="/content/oaizib_info")
import glob
for f in glob.glob(f"{info_dir}/**/*", recursive=True):
    if os.path.isfile(f) and f.split(".")[-1] in ("csv","json","txt"):
        print(f, os.path.getsize(f), "bytes")
# TODO: mo file metadata, tim cot KL + cot noi voi oaizib_XXX.
# Neu tim thay: ghi BSC_ROOT/splits/kl_map.json = {case_id: kl_grade}
# Neu khong: bo qua - bin do day la stratifier chinh.

## Gate 0 checklist
- [ ] `splits_zib_v1*.json` trên Drive
- [ ] B0/B1/B2 giải nén vào `BSC_ROOT/baselines/`
- [ ] **Metric femoral_cart khớp bảng đã công bố** ← quan trọng nhất
- [ ] KL map (tùy chọn)

**Không có gì quan trọng nằm ở `/content/`** (ngoài `/content/drive/`). ✅ → sang bsc_01.